# Funnel sur les contributions

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-08-01 00:00:00'
interval_stop = '2026-09-01 00:00:00'

In [ ]:
columns = ['action_id',
    'idvisit',
    'actions',
    'visitduration', 
    'operatingsystemname',
    'action_timestamp',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'referrertype', 
    'referrername',
    'experiments']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
        """

In [ ]:
query_visits =  range_query + f"""
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

In [ ]:
visits_df["action_timestamp"] = pd.to_datetime(visits_df["action_timestamp"], utc=True)

# Ordre intra-visite = suffixe de action_id ("<idvisit>_<n>").
# NB : la colonne `actions` = nombre total d'actions de la visite, PAS un rang.
visits_df["rank"] = pd.to_numeric(visits_df["action_id"].str.split("_").str[-1], errors="coerce")

print(f"rank non extrait : {visits_df['rank'].isna().sum()}")
print(f"action_id en double : {visits_df['action_id'].duplicated().sum()}")

visits_df = (
    visits_df.drop_duplicates("action_id")
    .sort_values(["idvisit", "rank"])
    .reset_index(drop=True)
)
assert not visits_df.duplicated(["idvisit", "rank"]).any(), "rank non unique par visite"

In [ ]:
from urllib.parse import urlsplit, urlunsplit

def clean_url(url):
    if pd.isna(url) or not url:
        return url
    parts = urlsplit(url)
    # on ne garde que scheme + host + path, sans query ni fragment
    cleaned = urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))
    # slash final (mais pas la racine "https://site/")
    if cleaned.endswith("/") and parts.path != "/":
        cleaned = cleaned[:-1]
    return cleaned

visits_df["action_url_clean"] = visits_df["action_url"].apply(clean_url)

On commence par regrouper par visites et filtrer sur les visites où l'on a consulté une contribution générique.

In [ ]:
base_slugs = [
    "quel-est-le-salaire-minimum",
    "quel-est-le-salaire-minimum-dun-alternant-en-2026",
    "le-preavis-de-licenciement-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "quelle-est-la-duree-du-conge-de-maternite",
    "dans-le-cadre-dun-cdd-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelle-est-la-duree-maximale-du-contrat-de-mission-interim",
    "quelle-est-la-duree-de-preavis-en-cas-de-depart-a-la-retraite",
    "si-le-salarie-est-malade-pendant-ses-conges-quelles-en-sont-les-consequences",
    "si-un-poste-se-libere-ou-est-cree-dans-lentreprise-lemployeur-doit-il-en-informer-les-salaries-ou-le-leur-proposer-en-priorite",
    "heures-supplementaires",
    "quelles-sont-les-conditions-de-cumul-demplois",
    "conges-supplementaires-pour-anciennete",
    "faut-il-respecter-un-delai-de-carence-entre-deux-cdd-si-oui-quelle-est-sa-duree",
    "faut-il-respecter-un-delai-de-carence-entre-deux-contrats-de-mission-interim",
    "les-conges-pour-evenements-familiaux",
    "lentreprise-peut-elle-embaucher-dans-le-cadre-dun-cdi-de-chantier-ou-doperation",
    "quelle-peut-etre-la-duree-maximale-dun-cdd",
    "comment-determiner-lanciennete-du-salarie",
    "a-quelles-indemnites-peut-pretendre-un-salarie-qui-part-a-la-retraite",
    "embauche-en-contrat-dextra-cdd-dusage",
    "quelles-informations-doivent-figurer-dans-le-contrat-de-travail-ou-la-lettre-dengagement",
    "en-cas-de-perte-de-marche-par-lemployeur-quelles-sont-les-conditions-dun-transfert-des-contrats-de-travail",
    "arret-maladie-pendant-la-periode-dessai-quelles-sont-les-regles",
    "quest-ce-quune-rupture-conventionnelle",
    "combien-de-fois-le-contrat-de-travail-peut-il-etre-renouvele",
    "arret-maladie-pendant-le-preavis-quelles-consequences",
    "dans-le-cadre-dun-contrat-de-mission-interim-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelles-sont-les-consequences-du-non-respect-du-preavis-par-le-salarie-ou-lemployeur",
    "quelle-est-la-duree-maximale-de-la-periode-dessai-sans-et-avec-renouvellement",
    "la-periode-dessai-peut-elle-etre-renouvelee",
    "quelles-sont-les-conditions-de-la-clause-de-non-concurrence",
    "quelle-est-la-duree-de-preavis-en-cas-de-mise-a-la-retraite",
    "travail-du-dimanche-quelle-contrepartie",
    "quelle-est-la-duree-du-preavis-en-cas-de-demission",
    "en-cas-de-maladie-le-salarie-a-t-il-droit-a-une-garantie-demploi",
    "quelles-sont-les-consequences-du-deces-de-lemployeur-sur-le-contrat-de-travail",
    "jours-feries-et-ponts-dans-le-secteur-prive",
    "quelle-est-la-duree-de-preavis-en-cas-de-licenciement",
    "est-il-obligatoire-davoir-un-contrat-de-travail-ecrit-et-signe",
    "le-preavis-de-demission-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "en-cas-darret-maladie-du-salarie-lemployeur-doit-il-assurer-le-maintien-de-salaire",
    "quelles-sont-les-conditions-dindemnisation-pendant-le-conge-de-maternite",
]



In [ ]:
from urllib.parse import urlsplit

# --- 1. Normalisation de l'URL (chemin seul, sans query/ancre, sans / final) ---
def clean_path(url):
    if not isinstance(url, str):
        return None
    return urlsplit(url).path.rstrip("/")

# On part de action_url_clean si présent, sinon action_url
url_col = "action_url_clean" if "action_url_clean" in visits_df.columns else "action_url"
visits_df["path"] = visits_df[url_col].map(clean_path)

# --- 2. Détection des page_views sur une contribution générique ---
generic_paths = {f"/contribution/{slug}" for slug in base_slugs}

visits_df["is_generic_contrib"] = (
    (visits_df["action_type"] == "action")
    & visits_df["path"].isin(generic_paths)
)

# --- 4. Groupement par visite + filtre ---
visits_with_generic = visits_df.groupby("idvisit")["is_generic_contrib"].any()
generic_idvisits = visits_with_generic[visits_with_generic].index

filtered_df = visits_df[visits_df["idvisit"].isin(generic_idvisits)]
grouped = filtered_df.groupby("idvisit")

print(f"Visites totales : {visits_df['idvisit'].nunique()}")
print(f"Visites avec contribution générique : {len(generic_idvisits)}")

In [ ]:
# --- 1. Première page vue de chaque visite ---
page_views = visits_df[visits_df["action_type"] == "action"]
first_pages = (
    page_views.sort_values(["idvisit", "rank"])
    .groupby("idvisit")
    .first()
)

# --- 2. Flag : la visite démarre sur une contribution générique ---
first_pages["starts_on_generic_contrib"] = first_pages["path"].isin(generic_paths)

# --- 3. Pourcentages ---
# a) Parmi toutes les visites
pct_all = first_pages["starts_on_generic_contrib"].mean() * 100

# b) Parmi les visites qui ont vu une contribution générique (ton filtre précédent)
first_pages_generic = first_pages.loc[first_pages.index.isin(generic_idvisits)]
pct_among_generic = first_pages_generic["starts_on_generic_contrib"].mean() * 100

print(f"Visites démarrant sur une contribution générique : {first_pages['starts_on_generic_contrib'].sum()}")
print(f"  → {pct_all:.1f} % de toutes les visites")
print(f"  → {pct_among_generic:.1f} % des visites ayant consulté une contribution générique")

In [ ]:
# --- 1. Tri intra-visite ---
df = visits_df[visits_df["idvisit"].isin(generic_idvisits)].copy()
df = df.sort_values(["idvisit", "rank"])

# --- 2. Découpage en segments : un nouveau segment à chaque page_view ---
df["is_page_view"] = df["action_type"] == "action"
df["segment_id"] = df.groupby("idvisit")["is_page_view"].cumsum()

# Chemin de la page qui a ouvert le segment, propagé aux events qui suivent
df["segment_path"] = df["path"].where(df["is_page_view"])
df["segment_path"] = df.groupby("idvisit")["segment_path"].ffill()

# --- 3. Ne garder que les segments sur une contribution générique ---
contrib_df = df[df["segment_path"].isin(generic_paths)].copy()
contrib_df["slug"] = contrib_df["segment_path"].str.replace("/contribution/", "", regex=False)

print(f"Lignes conservées : {len(contrib_df)} / {len(df)}")
print(f"Segments (page_view de contrib générique) : {contrib_df.groupby(['idvisit','segment_id']).ngroups}")
print(f"Visites : {contrib_df['idvisit'].nunique()}")

In [ ]:
first_seg = contrib_df.groupby("idvisit")["segment_id"].transform("min")
contrib_first_df = contrib_df[contrib_df["segment_id"] == first_seg]

In [ ]:
events = contrib_df[contrib_df["action_type"] == "event"]

distinct_events = (
    events.groupby(["action_eventcategory", "action_eventaction"])
    .agg(
        nb_events=("action_id", "size"),
        nb_visites=("idvisit", "nunique"),
        nb_names=("action_eventname", "nunique"),
    )
    .sort_values("nb_events", ascending=False)
    .reset_index()
)

pd.set_option("display.max_rows", 200)
distinct_events

In [ ]:
excluded_categories = ["nps", "selectrelated", "selectedsuggestion", "selectresult", "pagecc_searchcc", "feedback_simulateurs_rupture_co", "feedback_suggestion_rupture_co", "fiche-service-public", "search", "contact", "page_modeles_de_documents", "page_home", "modeles-de-courriers", "feedback", "accord_enterprise_search"]

cat_lower = contrib_df["action_eventcategory"].str.lower()
mask_excluded = (contrib_df["action_type"] == "event") & cat_lower.isin(excluded_categories)

# Détail par catégorie pour contrôle
print(cat_lower[mask_excluded].value_counts())
print(f"Total lignes supprimées : {mask_excluded.sum()}")

contrib_df = contrib_df[~mask_excluded].copy()

In [ ]:
events = contrib_df[contrib_df["action_type"] == "event"]

distinct_events = (
    events.groupby(["action_eventcategory"])
    .agg(
        nb_events=("action_id", "size"),
        nb_visites=("idvisit", "nunique"),
        nb_names=("action_eventname", "nunique"),
    )
    .sort_values("nb_events", ascending=False)
    .reset_index()
)

pd.set_option("display.max_rows", 200)
distinct_events

In [ ]:
seg_key = ["idvisit", "segment_id"]

# Nombre de lignes autres que le page_view (events + downloads) par segment
seg_activity = (
    contrib_df.assign(is_other=~contrib_df["is_page_view"])
    .groupby(seg_key)["is_other"]
    .sum()
    .rename("nb_actions")
)

# --- Niveau segment ---
nb_seg = len(seg_activity)
nb_seg_inactive = (seg_activity == 0).sum()
print(f"Segments contrib générique : {nb_seg}")
print(f"  sans aucune action : {nb_seg_inactive} ({nb_seg_inactive / nb_seg * 100:.1f} %)")

# --- Niveau visite : toutes les contribs vues sont restées inactives ---
visit_inactive = seg_activity.groupby("idvisit").max() == 0
nb_visits = len(visit_inactive)
nb_visits_inactive = visit_inactive.sum()
print(f"Visites : {nb_visits}")
print(f"  sans aucune action sur la/les contrib(s) : {nb_visits_inactive} ({nb_visits_inactive / nb_visits * 100:.1f} %)")

In [ ]:
first_actions = (
    contrib_df[~contrib_df["is_page_view"]]
    .sort_values(["idvisit", "rank"])
    .groupby(seg_key)
    .first()
)
nb_seg_active = len(first_actions)

In [ ]:
mask_p3 = (
    (contrib_df["action_type"] == "event")
    & (contrib_df["action_eventcategory"] == "cc_search_type_of_users")
    & (contrib_df["action_eventaction"] == "click_p3")
)

visits_p3 = contrib_df.loc[mask_p3, "idvisit"].unique()

print(f"Events click_p3 : {mask_p3.sum()}")
print(f"Visites avec click_p3 : {len(visits_p3)} ({len(visits_p3) / nb_visits * 100:.1f} % des visites contrib générique)")

In [ ]:
mask_ent_search = (
    (contrib_df["action_type"] == "event")
    & (contrib_df["action_eventcategory"] == "enterprise_search")
)

visits_ent_search = contrib_df.loc[mask_ent_search, "idvisit"].unique()

print(f"Events enterprise_search : {mask_ent_search.sum()}")
print(f"Visites avec enterprise_search : {len(visits_ent_search)}")
print(f"  → {len(visits_ent_search) / nb_visits * 100:.1f} % de toutes les visites contrib générique")

# Croisement avec click_p3 (recherche d'entreprise = parcours P2 normalement)
both = set(visits_ent_search) & set(visits_p3)
print(f"  → dont {len(both)} ont aussi fait click_p3 ({len(both) / len(visits_ent_search) * 100:.1f} %)")

In [ ]:
def count_event(df, first_actions, category, action=None, nb_visits=nb_visits):
    mask = (df["action_type"] == "event") & (df["action_eventcategory"] == category)
    mask_first = (first_actions["action_type"] == "event") & (first_actions["action_eventcategory"] == category)
    if action is not None:
        mask &= df["action_eventaction"] == action
        mask_first &= first_actions["action_eventaction"] == action

    visits_any = df.loc[mask, "idvisit"].unique()
    visits_first = first_actions[mask_first].index.get_level_values("idvisit").unique()

    label = f"{category} / {action}" if action else category
    print(f"=== {label} ===")
    print(f"Events : {mask.sum()}")
    print(f"Visites avec l'event : {len(visits_any)} ({len(visits_any) / nb_visits * 100:.1f} % des visites contrib)")
    print(f"Segments avec l'event en 1re action : {mask_first.sum()} ({mask_first.sum() / len(first_actions) * 100:.1f} % des segments actifs)")
    print(f"Visites avec l'event en 1re action : {len(visits_first)} "
          f"({len(visits_first) / max(len(visits_any), 1) * 100:.1f} % des visites avec l'event, "
          f"{len(visits_first) / nb_visits * 100:.1f} % des visites contrib)\n")
    return visits_any, visits_first

visits_p1, visits_p1_first = count_event(contrib_df, first_actions, "cc_search_type_of_users", "click_p1")
visits_p2, visits_p2_first = count_event(contrib_df, first_actions, "cc_search_type_of_users", "click_p2")
visits_p3, visits_p3_first = count_event(contrib_df, first_actions, "cc_search_type_of_users", "click_p3")

In [ ]:
contrib_actions = [
    "click_afficher_les_informations_CC",
    "click_afficher_les_informations_générales",
    "click_afficher_les_informations_sans_CC",
]

results_contrib = {
    action: count_event(contrib_df, first_actions, "contribution", action)
    for action in contrib_actions
}

In [ ]:
# --- Clés d'events ---
K_P1        = "cc_search_type_of_users / click_p1"
K_P2        = "cc_search_type_of_users / click_p2"
K_P3        = "cc_search_type_of_users / click_p3"
K_ENT       = "enterprise_search"
K_AFF_CC    = "contribution / click_afficher_les_informations_CC"
K_AFF_GEN   = "contribution / click_afficher_les_informations_générales"
K_AFF_SANS  = "contribution / click_afficher_les_informations_sans_CC"

ev = contrib_df[contrib_df["action_type"] == "event"].copy()
ev["key"] = ev["action_eventcategory"] + " / " + ev["action_eventaction"]
ev.loc[ev["action_eventcategory"] == K_ENT, "key"] = K_ENT   # toutes les actions de enterprise_search


def first_after(ev, keys, after=None):
    """Position (rank) de la 1re occurrence de l'un des `keys` par segment,
    strictement après la position `after` (Series indexée par seg_key)."""
    sub = ev[ev["key"].isin(keys)]
    if after is not None:
        sub = sub.join(after.rename("_after"), on=seg_key, how="inner")
        sub = sub[sub["rank"] > sub["_after"]]
    return sub.groupby(seg_key)["rank"].min()


def build_funnel(name, steps):
    """steps = liste de (label, [keys]) ; chaque étape doit suivre la précédente."""
    rows, prev = [], None
    for label, keys in steps:
        pos = first_after(ev, keys, after=prev)
        rows.append((label, pos.index.get_level_values("idvisit").nunique(), len(pos)))
        prev = pos
    out = pd.DataFrame(rows, columns=["etape", "visites", "segments"])
    out.insert(0, "parcours", name)
    return out


funnels = pd.concat([
    build_funnel("P1 - CC directe", [
        ("Sélection CC (click_p1)",       [K_P1]),
        ("Affichage réponse",             [K_AFF_CC, K_AFF_GEN]),
    ]),
    build_funnel("P2 - Entreprise", [
        ("Recherche entreprise",          [K_ENT]),
        ("Sélection entreprise/CC (click_p2)", [K_P2]),
        ("Affichage réponse",             [K_AFF_CC, K_AFF_GEN]),
    ]),
    build_funnel("P3 - Sans CC", [
        ("Choix sans CC (click_p3)",      [K_P3]),
        ("Affichage réponse sans CC",     [K_AFF_SANS]),
    ]),
], ignore_index=True)

# --- Pertes ---
funnels["% base contrib"] = (funnels["visites"] / nb_visits * 100).round(1)
funnels["% etape prec."] = (
    funnels["visites"] / funnels.groupby("parcours")["visites"].shift(1) * 100
).round(1)
funnels["perte"] = funnels.groupby("parcours")["visites"].shift(1) - funnels["visites"]

print(f"Base : {nb_visits} visites sur une contribution générique\n")
funnels

In [ ]:
# Visites entrant dans au moins un parcours
entry_keys = [K_P1, K_ENT, K_P3]
visits_any_entry = set(first_after(ev, entry_keys).index.get_level_values("idvisit"))
nb_lost_global = nb_visits - len(visits_any_entry)
print(f"Visites sans aucun parcours : {nb_lost_global} ({nb_lost_global / nb_visits * 100:.1f} %)")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def plot_funnels(funnels, nb_visits, nb_lost_global, nb_inactive=None,
                 box_w=2.6, box_h=1.1, gap=1.6, row_h=2.6):
    parcours = list(funnels["parcours"].unique())
    n_rows = len(parcours)
    n_steps_max = funnels.groupby("parcours").size().max()
    x_start = box_w + gap * 1.6          # décalage des parcours à droite de la boîte de base

    box_color, box_edge = "#DCE6F2", "#4A6FA5"
    base_color, base_edge = "#EEF1F5", "#8A9BB0"
    ink, ink_muted, loss_ink = "#1F2933", "#616E7C", "#9F3A38"
    fmt = lambda n: f"{n:,.0f}".replace(",", " ")

    fig, ax = plt.subplots(figsize=(x_start + n_steps_max * (box_w + gap) + 0.5, n_rows * row_h + 1))
    ax.set_xlim(-0.3, x_start + n_steps_max * (box_w + gap))
    ax.set_ylim(-0.8, n_rows * row_h)
    ax.axis("off")

    def draw_box(x, y, label, visits, sub_label=None, base=False):
        ax.add_patch(FancyBboxPatch((x, y), box_w, box_h,
                                    boxstyle="round,pad=0,rounding_size=0.12",
                                    facecolor=base_color if base else box_color,
                                    edgecolor=base_edge if base else box_edge, linewidth=1.5))
        ax.text(x + box_w / 2, y + box_h * 0.72, label, ha="center", va="center", fontsize=8.5, color=ink)
        ax.text(x + box_w / 2, y + box_h * 0.30, f"{fmt(visits)} visites",
                ha="center", va="center", fontsize=11, fontweight="bold", color=ink)
        if sub_label:
            ax.text(x + box_w / 2, y - 0.22, sub_label, ha="center", va="center", fontsize=8, color=ink_muted)

    def draw_arrow(p0, p1, pct_kept, lost, curved=False):
        ax.annotate("", xy=p1, xytext=p0,
                    arrowprops=dict(arrowstyle="-|>", color=ink_muted, lw=1.5,
                                    connectionstyle="arc3,rad=0.15" if curved else "arc3,rad=0"))
        mx, my = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
        ax.text(mx, my + 0.18, f"{pct_kept:.1f} %", ha="center", va="bottom", fontsize=9, color=ink_muted)
        if lost is not None:
            ax.text(mx, my - 0.18, f"−{fmt(lost)} ({100 - pct_kept:.1f} %)",
                    ha="center", va="top", fontsize=9, color=loss_ink)

    # --- Boîte de base, centrée verticalement ---
    y_base = (n_rows - 1) * row_h / 2
    sub = f"{fmt(nb_lost_global)} sans aucun parcours ({nb_lost_global / nb_visits * 100:.1f} %)"
    if nb_inactive is not None:
        sub += f"\ndont {fmt(nb_inactive)} sans action ({nb_inactive / nb_visits * 100:.1f} %)"
    draw_box(0, y_base, "Visites contrib générique", nb_visits, sub, base=True)

    # --- Parcours ---
    for r, name in enumerate(parcours):
        y = (n_rows - 1 - r) * row_h
        steps = funnels[funnels["parcours"] == name].reset_index(drop=True)
        ax.text(x_start, y + box_h + 0.35, name, fontsize=11, fontweight="bold", color=ink, va="center")

        # flèche base → entrée du parcours (pas de perte affichée : les parcours ne sont pas exclusifs)
        first = steps.loc[0]
        draw_arrow((box_w, y_base + box_h / 2), (x_start, y + box_h / 2),
                   first["% base contrib"], None, curved=True)

        for i, row in steps.iterrows():
            x = x_start + i * (box_w + gap)
            draw_box(x, y, row["etape"], row["visites"], f"{row['% base contrib']:.1f} % de la base")
            if i < len(steps) - 1:
                nxt = steps.loc[i + 1]
                draw_arrow((x + box_w, y + box_h / 2), (x + box_w + gap, y + box_h / 2),
                           nxt["% etape prec."], nxt["perte"])

    fig.suptitle(f"Funnel contributions génériques — base : {fmt(nb_visits)} visites",
                 fontsize=13, color=ink, x=0.01, ha="left")
    fig.tight_layout()
    return fig

fig = plot_funnels(funnels, nb_visits, nb_lost_global, nb_inactive=nb_visits_inactive)
plt.show()

In [ ]:
MOBILE_OS  = {"iOS", "Android", "iPadOS", "Windows Phone", "HarmonyOS", "KaiOS"}
DESKTOP_OS = {"Windows", "Mac", "GNU/Linux", "Ubuntu", "Chrome OS", "Fedora", "Debian"}

def device_type(os_name):
    if os_name in MOBILE_OS:
        return "mobile"
    if os_name in DESKTOP_OS:
        return "desktop"
    return "autre"

device_by_visit = (
    contrib_df.drop_duplicates("idvisit").set_index("idvisit")["operatingsystemname"].map(device_type)
)
print(device_by_visit.value_counts())


def compute_funnel(idvisits):
    """Funnel + indicateurs globaux restreints à un ensemble de visites."""
    ev_sub = ev[ev["idvisit"].isin(idvisits)]
    n = len(idvisits)

    def first_after_sub(keys, after=None):
        sub = ev_sub[ev_sub["key"].isin(keys)]
        if after is not None:
            sub = sub.join(after.rename("_after"), on=seg_key, how="inner")
            sub = sub[sub["rank"] > sub["_after"]]
        return sub.groupby(seg_key)["rank"].min()

    def build(name, steps):
        rows, prev = [], None
        for label, keys in steps:
            pos = first_after_sub(keys, after=prev)
            rows.append((label, pos.index.get_level_values("idvisit").nunique(), len(pos)))
            prev = pos
        out = pd.DataFrame(rows, columns=["etape", "visites", "segments"])
        out.insert(0, "parcours", name)
        return out

    f = pd.concat([
        build("P1 - CC directe", [("Sélection CC (click_p1)", [K_P1]),
                                  ("Affichage réponse", [K_AFF_CC, K_AFF_GEN])]),
        build("P2 - Entreprise", [("Recherche entreprise", [K_ENT]),
                                  ("Sélection entreprise/CC (click_p2)", [K_P2]),
                                  ("Affichage réponse", [K_AFF_CC, K_AFF_GEN])]),
        build("P3 - Sans CC",    [("Choix sans CC (click_p3)", [K_P3]),
                                  ("Affichage réponse sans CC", [K_AFF_SANS])]),
    ], ignore_index=True)
    f["% base contrib"] = (f["visites"] / n * 100).round(1)
    f["% etape prec."] = (f["visites"] / f.groupby("parcours")["visites"].shift(1) * 100).round(1)
    f["perte"] = f.groupby("parcours")["visites"].shift(1) - f["visites"]

    any_entry = set(first_after_sub([K_P1, K_ENT, K_P3]).index.get_level_values("idvisit"))
    inactive = (seg_activity.groupby("idvisit").max() == 0)
    return {
        "funnels": f,
        "nb_visits": n,
        "nb_lost_global": n - len(any_entry),
        "nb_inactive": int(inactive[inactive.index.isin(idvisits)].sum()),
    }


for device in ["mobile", "desktop"]:
    ids = device_by_visit[device_by_visit == device].index
    r = compute_funnel(ids)
    fig = plot_funnels(r["funnels"], r["nb_visits"], r["nb_lost_global"], nb_inactive=r["nb_inactive"])
    fig.suptitle(f"Funnel contributions génériques — {device} ({r['nb_visits']:,.0f} visites)".replace(",", " "),
                 fontsize=13, x=0.01, ha="left")
    plt.show()

In [ ]:
cmp = pd.concat(
    {d: compute_funnel(device_by_visit[device_by_visit == d].index)["funnels"]
        .set_index(["parcours", "etape"])[["visites", "% base contrib", "% etape prec."]]
     for d in ["mobile", "desktop"]},
    axis=1,
)
cmp